#### Importaçao de Bibliotecas

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import json

#### Fase 1: Processo de extração e Integração

In [102]:
#Leitura excel
df_metas = pd.read_excel('metas_vendas_2024.xlsx')
    #print(df_metas)

#Leitura csv
df_satisfacao = pd.read_csv('pesquisa_satisfacao_clientes.csv')
    #print(df_satisfacao.head())

#DataFrame de satisfação média por Filial. (Colocado aqui para melhor organização)
df_satisfacao_filial = df_satisfacao.groupby('Filial')['Nota'].mean().reset_index()
df_satisfacao_filial.rename(columns={'Nota': 'Satisfacao_Media'}, inplace=True)
    #print(df_satisfacao_filial)


#Leitura Json / Criação da tabela Filial
df_custo = pd.read_json('custos_operacionais.json').T.reset_index()
df_custo.rename(columns={'index': 'Filial'}, inplace=True)
    #print(df_custo)



#Leitura do Banco de dados
connection = sqlite3.connect('DataOrganized.db')
query = """
SELECT v.id_venda, v.data_venda, v.vendedor, v.filial, p.nome_produto, p.categoria, v.quantidade, v.preco_venda, p.preco_tabela
FROM vendas v
JOIN produtos p ON v.id_produto = p.id_produto;
"""
df_vendas_produtos = pd.read_sql_query(query, connection)
connection.close()
df_vendas_produtos.rename(columns={'filial': 'Filial'}, inplace=True)#estava diferente dos outros dataframes
    #print(df_vendas_produtos.head())




##### Merge dos Dados

In [ ]:
# Merge vendas/produtos com metas de faturamento
df_vendas_metas = pd.merge(df_vendas_produtos, df_metas, on='Filial', how='left')
    #print(df_vendas_metas.head())

# Merge anterior com custos operacionais
df_vendas_metas_custos = pd.merge(df_vendas_metas, df_custo, on='Filial', how='left')
    #print(df_vendas_metas_custos.head())

# Merge final
df_final = pd.merge(df_vendas_metas_custos, df_satisfacao_filial, on='Filial', how='left')
    #print(df_final.head())


df_final.rename(columns= {'preco_tabela' : 'Preco_Venda'}, inplace=True)
    #print(df_final.head())
df_final.rename(columns = {'preco_venda': 'Preco_Tabela'}, inplace=True)
    #print(df_final.head())
#Obs: O preço de tabela estava mais alto que o preço de venda


#### Fase 2 : Engenharia de Atributos (Cálculos de Negócio)

In [ ]:
# Calculos com pandas(Dataframe)/Criando as tabelas novas

df_final['Faturamento_Real'] = df_final['quantidade'] * df_final['Preco_Venda']
    #print(df_final.head())

df_final['Margem_Bruta'] = (df_final['Preco_Venda'] - df_final['Preco_Tabela'])/ df_final['Preco_Venda']
    #print(df_final.head())

df_final['Atingimento_Meta'] = df_final['Faturamento_Real'] / df_final['Meta_Faturamento']
    #print(df_final.head())

df_final['Custo_Total'] = df_final[['Aluguel', 'Energia', 'Marketing', 'Logistica']].sum(axis=1)
    #print(df_final.head())

df_final['Lucro_Liquido'] = df_final['Faturamento_Real'] - df_final['Custo_Total']
    print(df_final.head())


#### Fase 3 : Diagnóstico Estatístico Profundo (Para Cada Filial) 

In [105]:
# Utilizei dois metodos diferentes para fazer.(Pandas e Numpy)

#Pandas
medias = df_final.groupby('Filial')[['Margem_Bruta', 'Lucro_Liquido', 'Satisfacao_Media']].mean()
print('Médias por filial:\n', medias)

medianas = df_final.groupby('Filial')[['Margem_Bruta', 'Lucro_Liquido', 'Satisfacao_Media']].median()
print('\nMedianas por filial:\n', medianas)

desvios = df_final.groupby('Filial')[['Margem_Bruta', 'Lucro_Liquido', 'Satisfacao_Media']].std()
print('\nDesvios padrão por filial:\n', desvios)


#Numpy dentro de um for

for Filial, grupo in df_final.groupby('Filial'):
    print(f'--- Estatísticas para a Filial: {Filial} (numpy) ---')
    print(f"Média (Margem Bruta): {np.mean(grupo['Margem_Bruta']):.4f}")
    print(f"Mediana (Margem Bruta): {np.median(grupo['Margem_Bruta']):.4f}")
    print(f"Desvio Padrão (Margem Bruta): {np.std(grupo['Margem_Bruta'], ddof=1):.4f}")
    print(f"Média (Lucro Líquido): {np.mean(grupo['Lucro_Liquido']):.2f}")
    print(f"Mediana (Lucro Líquido): {np.median(grupo['Lucro_Liquido']):.2f}")
    print(f"Desvio Padrão (Lucro Líquido): {np.std(grupo['Lucro_Liquido'], ddof=1):.2f}")
    print(f"Média (Satisfação): {np.mean(grupo['Satisfacao_Media']):.2f}")
    print(f"Mediana (Satisfação): {np.median(grupo['Satisfacao_Media']):.2f}")
    print(f"Desvio Padrão (Satisfação): {np.std(grupo['Satisfacao_Media'], ddof=1):.2f}")
    print()


Médias por filial:
         Margem_Bruta  Lucro_Liquido  Satisfacao_Media
Filial                                               
MG          0.045978  -16041.864407              4.78
RJ          0.340870  -24593.800000              1.76
SP          0.052367  -29295.121951              4.66

Medianas por filial:
         Margem_Bruta  Lucro_Liquido  Satisfacao_Media
Filial                                               
MG          0.042340       -16950.0              4.78
RJ          0.328187       -28275.0              1.76
SP          0.051833       -29950.0              4.66

Desvios padrão por filial:
         Margem_Bruta  Lucro_Liquido  Satisfacao_Media
Filial                                               
MG          0.030455    2544.372068               0.0
RJ          0.089037   11533.043369               0.0
SP          0.031125    1623.095995               0.0
--- Estatísticas para a Filial: MG (numpy) ---
Média (Margem Bruta): 0.0460
Mediana (Margem Bruta): 0.0423
Desvio Padr